# MCP server 2 — IoT and asset registry

Compare installed sensors in the registry with measured sensors in telemetry.

**Tutorial contract:** run cells from top to bottom. Every external dependency is checked before use,
outputs go under `artifacts/kdd_tutorial/`, and no credential value is printed.


## Goal

Walk from site → asset → installed/measured sensor inventory.

**Requires:** seeded CouchDB (`docker compose -f src/couchdb/docker-compose.yaml up -d`)


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ENV_FILE = REPO / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(f"Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.")
load_dotenv(ENV_FILE, override=True)
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("environment source:", ENV_FILE)
print("python:", sys.version.split()[0])


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

ENTRY_POINTS = {
    "iot": "iot-mcp-server", "utilities": "utilities-mcp-server",
    "fmsr": "fmsr-mcp-server", "wo": "wo-mcp-server",
    "tsfm": "tsfm-mcp-server", "vibration": "vibration-mcp-server",
}

async def mcp_session(server, operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), ENTRY_POINTS[server]],
        cwd=str(REPO),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def text_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text

async def list_tools(server):
    response = await mcp_session(server, "list")
    return [{"name": t.name, "description": t.description, "schema": t.inputSchema} for t in response.tools]

async def call_tool(server, name, **arguments):
    return text_result(await mcp_session(server, "call", name, arguments))


## 1. Discover the live MCP contract

This starts the real stdio server and asks it for its tool schemas.


In [ ]:
tools = await list_tools("iot")
[(t["name"], list(t["schema"].get("properties", {}))) for t in tools]

## 2. CouchDB preflight


In [ ]:
import socket
from urllib.parse import urlparse

def tcp_reachable(url, timeout=1.0):
    parsed = urlparse(url)
    host, port = parsed.hostname or "localhost", parsed.port or 5984
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

COUCHDB_URL = os.getenv("COUCHDB_URL", "http://localhost:5984")
print("CouchDB reachable:", tcp_reachable(COUCHDB_URL), "at", COUCHDB_URL)


In [ ]:
assert tcp_reachable(COUCHDB_URL), "Start CouchDB: docker-compose -f src/couchdb/docker-compose.yaml up -d"


## 3. Traverse the asset hierarchy


In [ ]:
sites = await call_tool("iot", "sites")
sites


In [ ]:
assets = await call_tool("iot", "assets", site_name="MAIN")
assets

In [ ]:
site = "MAIN"

assets_result = await call_tool(
    "iot",
    "assets",
    site_name=site,
)

asset = assets_result["assets"][0]["asset_id"]
print("Using asset:", asset)

measured = await call_tool(
    "iot",
    "measured_sensors",
    site_name=site,
    asset_id=asset,
)

installed = await call_tool(
    "iot",
    "installed_sensors",
    site_name=site,
    asset_id=asset,
)

detail = await call_tool(
    "iot",
    "asset_detail",
    site_name=site,
    asset_id=asset,
)

{
    "detail": detail,
    "measured": measured,
    "installed": installed,
}

## Takeaway

You exercised the server through MCP JSON-RPC over stdio—the same boundary the agents use.
